**Purpose:** Frozen copy of the exact code that produced the final Reddit run h5a8a3d210b.

**Inputs:** `/kaggle/input/datasets/hugoverssimo/reddit-full/reddit_0.parquet`

**Notes:** paths resolve via `src.config` (run `pip install -e .` from the repo root first).


https://www.kaggle.com/code/hugoverssimo/fork-of-full-reddit-dataset-run?scriptVersionId=303624339

In [ ]:
BUCKET = 1

In [ ]:
# ===== Cell 1 (Kaggle-safe) =====
# Fixes NumPy/SciPy/sklearn binary compatibility + keeps your parquet stack stable.

!pip -q uninstall -y numpy scipy scikit-learn pandas pyarrow

# Pin to a known-good combo for Python 3.12 wheels on Kaggle
!pip -q install --no-cache-dir --force-reinstall \
  "numpy==2.0.2" \
  "scipy==1.14.1" \
  "scikit-learn==1.5.2" \
  "pandas==2.2.3" \
  "pyarrow==18.1.0"

!pip -q install flash-attn

# HF stack
!pip -q install -U --no-cache-dir \
  "transformers>=4.48.0,<5.3.0" \
  "accelerate>=0.34.0" \
  datasets evaluate

import numpy, scipy, sklearn, pandas, pyarrow
print("numpy", numpy.__version__)
print("scipy", scipy.__version__)
print("sklearn", sklearn.__version__)
print("pandas", pandas.__version__)
print("pyarrow", pyarrow.__version__)

In [ ]:
import torch
import torch.nn as nn
from transformers import AutoModel
import os
import json
import torch
from transformers import AutoTokenizer
from safetensors.torch import load_file

class ModernBERTMultiHead(nn.Module):
    def __init__(self, model_id, num_sectors, num_classes, dropout=0.1):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_id)
        hidden = self.encoder.config.hidden_size
        self.drop = nn.Dropout(dropout)
        self.heads = nn.ModuleList([
            nn.Linear(hidden, num_classes) for _ in range(num_sectors)
        ])

    def masked_mean_pool(self, last_hidden_state, attention_mask):
        mask = attention_mask.unsqueeze(-1).float()
        summed = (last_hidden_state * mask).sum(dim=1)
        denom = mask.sum(dim=1).clamp(min=1e-6)
        return summed / denom

    def forward(self, input_ids=None, attention_mask=None):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        pooled = self.masked_mean_pool(out.last_hidden_state, attention_mask)
        pooled = self.drop(pooled)
        logits = torch.stack([head(pooled) for head in self.heads], dim=1)
        return {"logits": logits}


MODEL_DIR = "/kaggle/input/models/hugoverssimo/reddit-5a8a3d210b/transformers/default/1"

# load metadata
with open(os.path.join(MODEL_DIR, "model_metadata.json")) as f:
    meta = json.load(f)

sector_cols = meta["sector_cols"]
sentiments = meta["sentiments"]
id2sent = {i: s for i, s in enumerate(sentiments)}

# tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)

# model
model = ModernBERTMultiHead(
    model_id=meta["model_id"],
    num_sectors=meta["num_sectors"],
    num_classes=meta["num_classes"],
    dropout=meta["hparams"]["dropout"],
)

state_dict = load_file(f"{MODEL_DIR}/model.safetensors")

model.load_state_dict(state_dict)
model.eval()

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

def predict(text,
            model=model,
            tokenizer=tokenizer,
            sector_cols=sector_cols,
            id2sent=id2sent,
            max_len=meta["hparams"]["max_len"],
            device=device):

    enc = tokenizer(
        text,
        truncation=True,
        max_length=max_len,
        padding=True,
        return_tensors="pt",
    )

    enc = {k: v.to(device) for k, v in enc.items()}

    with torch.no_grad():
        logits = model(**enc)["logits"]
        probs = torch.softmax(logits, dim=-1)[0]
        pred_ids = probs.argmax(dim=-1)

    predictions = {}

    for i, sector in enumerate(sector_cols):
        pred_id = int(pred_ids[i].cpu().item())
        sentiment = id2sent[pred_id]
        probability = float(probs[i, pred_id].cpu().item())

        predictions[sector] = (sentiment, round(probability,3))

    return predictions

In [ ]:
import pandas as pd
import json
import html
from datetime import datetime

reddit_posts = pd.read_parquet("/kaggle/input/datasets/hugoverssimo/reddit-full/reddit_0.parquet")
reddit_posts["top_comments"] = reddit_posts["top_comments"].apply(json.loads)
reddit_posts["submission"] = reddit_posts["submission"].apply(json.loads)

def digit_sum(n):
    return 1 + (n - 1) % 9 if n > 0 else 0

def create_reddit_post(reddit_df):

    reddit_to_label = {}

    for i, row in reddit_df.iterrows():

        to_label_text = "[POST]\n"
        submission = row["submission"]

        to_label_text += f"Title:\n{html.unescape(submission['title'])}\n"
        to_label_text += f"Body:\n{html.unescape(submission['selftext'])}\n\n"

        to_label_text += "[COMMENTS]\n"
        comments = row["top_comments"]

        if not comments:
            to_label_text += "No comments available.\n"
        else:
            top_comments = [(com["body"], com["score"]) for com in comments]
            top_comments = sorted(top_comments, key=lambda x: x[1], reverse=True)
            comments = [com[0] for com in top_comments]

            com_count, char_count = 0, 0
            for comment in comments:
                clean_comment = html.unescape(comment.replace("\n", "; ").strip())
                to_label_text += f"Comment {com_count + 1}:\n"
                to_label_text += f"{clean_comment}\n"

                com_count += 1
                char_count += len(clean_comment)

                #if com_count >= 5 and char_count >= 500:
                if com_count >= 10 and char_count >= 1000:
                    break

        reddit_to_label[i] = {
            "thread": to_label_text,
            "subreddit": row["subreddit"],
            "timestamp_utc": datetime.utcfromtimestamp(int(row["submission"]["created_utc"])),
            "hash": row["submission"]["created_utc"]
        }

    return pd.DataFrame.from_dict(reddit_to_label, orient="index", columns=["thread", "subreddit", "timestamp_utc", "hash"])
    
reddit_posts = create_reddit_post(reddit_posts)

In [ ]:
import os
import pandas as pd
import time

MAX_RUNTIME = 10 * 60 * 60   # 10 hours in seconds
start_time = time.time()

PROGRESS_FILE = f"bucket_{BUCKET}_progress.txt"

# ----------------------------
# Resume support: start after last processed id
# ----------------------------
start_after_id = None


processed_count = 0
reddit_labels = {}
started = start_after_id is None

for i, row in reddit_posts.iterrows():
    #if time.time() - start_time > MAX_RUNTIME:
    #    print("10 hour limit reached, stopping...")
    #    break

    
    current_id = str(i)

    if not started:
        if current_id == start_after_id:
            started = True
        continue

    #try:
    #    bucket = digit_sum(int(row["hash"]))
    #except Exception:
    #    print("bucket error", row["hash"])
    #    bucket = 1

    if True:#bucket == BUCKET:
        prediction = predict(row["thread"])

        reddit_labels[current_id] = {
            "subreddit": row["subreddit"],
            "timestamp_utc": row["timestamp_utc"]
        }

        for sector, label in prediction.items():
            reddit_labels[current_id][sector] = label

        processed_count += 1
        last_id_done = current_id


# ----------------------------
# Save outputs if anything was processed
# ----------------------------
import json
import pandas as pd

output = pd.DataFrame.from_dict(reddit_labels, orient="index")

label_cols = [c for c in output.columns if c.startswith("label_")]

for col in label_cols:
    output[col] = output[col].apply(
        lambda x: json.dumps({"label": x[0], "score": x[1]}) if isinstance(x, tuple)
        else None
    ).astype("string")

# Optional: make other object cols parquet-safe too
obj_cols = output.select_dtypes(include=["object"]).columns
output[obj_cols] = output[obj_cols].astype("string")

parquet_file = f"bucket_{BUCKET}_{last_id_done}.parquet"
output.to_parquet(parquet_file, index=True)


with open(PROGRESS_FILE, "w") as f:
    f.write(f"total_processed={processed_count}\n")
    f.write(f"last_id={last_id_done}\n")
    f.write(f"time={time.time() - start_time}")

print(f"Saved: {parquet_file}")
print(f"Progress saved in: {PROGRESS_FILE}")

In [ ]:
output